# Обучение YOLOv8n-OBB на стерео-датасете коробок

## Перед запуском
1. **Runtime → Change runtime type → GPU** (T4 хватит — это бесплатный тариф)
2. Загрузи файл `dataset_stereo_final.tar.gz` в Colab через панель Files слева (лежит локально в `experiments/`)
3. Запускай ячейки по порядку

## План
Один прогон на **100 эпох** без early-stopping, с чекпойнтами каждые 10 эпох.
После тренировки — отдельные eval'ы на `best.pt` (оптимум) и `last.pt` (переобучение) на val и test.

Ожидаемое время: ~30-45 минут на T4.

## 1. Установка

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Распаковка датасета
Убедись, что `dataset_stereo_final.tar.gz` загружен в `/content/`.

In [ ]:
import os
assert os.path.exists('/content/dataset_stereo_final.tar.gz'), \
    'Загрузи dataset_stereo_final.tar.gz в /content/ через панель Files слева'

!tar -xzf /content/dataset_stereo_final.tar.gz -C /content/
!ls /content/dataset_stereo_final/
print('---')
for split in ('train', 'val', 'test'):
    n = len(os.listdir(f'/content/dataset_stereo_final/images/{split}'))
    print(f'{split}: {n} картинок')

## 3. Правка путей в dataset.yaml под Colab

In [ ]:
yaml_text = '''path: /content/dataset_stereo_final
train: images/train
val: images/val
test: images/test
nc: 1
names: ['box']
'''
with open('/content/dataset_stereo_final/dataset.yaml', 'w') as f:
    f.write(yaml_text)
print(yaml_text)

## 4. Обучение

**Ключевое:**
- `epochs=100, patience=100` — без early-stop, прогоняем до конца чтобы увидеть переобучение
- `save_period=10` — чекпойнты epoch10.pt … epoch100.pt
- Автоматически сохраняются `best.pt` (оптимум по val) и `last.pt` (финальная переобученная)

**Аугментация — под top-down камеру + grayscale + OBB:**
- `mosaic=1.0, mixup=0.1, copy_paste=0.3` — регуляризация для малого датасета
- `fliplr=0.5, flipud=0.5` — сверху нет верха/низа
- `degrees=15` — вариации углов коробок
- `translate=0.1, scale=0.5` — позиция и размер
- `hsv_h=0, hsv_s=0` — отключены (grayscale)
- `hsv_v=0.2` — небольшая вариация яркости
- `perspective=0, shear=0` — камера фиксирована

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-obb.pt')

results = model.train(
    data='/content/dataset_stereo_final/dataset.yaml',
    epochs=100,
    patience=100,
    save_period=10,
    imgsz=640,
    batch=16,
    name='stereo_v1',
    device=0,
    plots=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.3,
    degrees=15,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    perspective=0.0,
    shear=0.0,
)

## 5. best.pt — val (эпоха с минимальным val loss)

In [ ]:
best_model = YOLO('runs/obb/stereo_v1/weights/best.pt')
m_best_val = best_model.val(
    data='/content/dataset_stereo_final/dataset.yaml',
    split='val',
    name='eval_best_val',
    plots=True,
)
print('--- best.pt on VAL ---')
print(f'mAP50:    {m_best_val.box.map50:.4f}')
print(f'mAP50-95: {m_best_val.box.map:.4f}')
print(f'Precision:{m_best_val.box.mp:.4f}')
print(f'Recall:   {m_best_val.box.mr:.4f}')

## 6. best.pt — TEST (unseen данные для защиты)

In [ ]:
m_best_test = best_model.val(
    data='/content/dataset_stereo_final/dataset.yaml',
    split='test',
    name='eval_best_test',
    plots=True,
)
print('--- best.pt on TEST ---')
print(f'mAP50:    {m_best_test.box.map50:.4f}')
print(f'mAP50-95: {m_best_test.box.map:.4f}')
print(f'Precision:{m_best_test.box.mp:.4f}')
print(f'Recall:   {m_best_test.box.mr:.4f}')

## 7. last.pt — демонстрация переобучения

In [ ]:
last_model = YOLO('runs/obb/stereo_v1/weights/last.pt')

m_last_val = last_model.val(
    data='/content/dataset_stereo_final/dataset.yaml',
    split='val',  name='eval_last_val',  plots=True,
)
m_last_test = last_model.val(
    data='/content/dataset_stereo_final/dataset.yaml',
    split='test', name='eval_last_test', plots=True,
)

print('=== best.pt vs last.pt ===')
header = f'{"":<15}{"best (val)":>12}{"last (val)":>12}{"best (test)":>14}{"last (test)":>14}'
print(header)
print('-' * len(header))
for name, bv, lv, bt, lt in [
    ('mAP50',     m_best_val.box.map50, m_last_val.box.map50, m_best_test.box.map50, m_last_test.box.map50),
    ('mAP50-95',  m_best_val.box.map,   m_last_val.box.map,   m_best_test.box.map,   m_last_test.box.map),
    ('Precision', m_best_val.box.mp,    m_last_val.box.mp,    m_best_test.box.mp,    m_last_test.box.mp),
    ('Recall',    m_best_val.box.mr,    m_last_val.box.mr,    m_best_test.box.mr,    m_last_test.box.mr),
]:
    print(f'{name:<15}{bv:>12.4f}{lv:>12.4f}{bt:>14.4f}{lt:>14.4f}')

## 8. Показать основные графики

In [ ]:
from IPython.display import Image, display
import os

for title, path in [
    ('Training curves (results.png)',          'runs/obb/stereo_v1/results.png'),
    ('best.pt — confusion matrix (val)',       'runs/obb/eval_best_val/confusion_matrix_normalized.png'),
    ('best.pt — F1 curve (val)',               'runs/obb/eval_best_val/BoxF1_curve.png'),
    ('best.pt — PR curve (val)',               'runs/obb/eval_best_val/BoxPR_curve.png'),
    ('best.pt — confusion matrix (TEST)',      'runs/obb/eval_best_test/confusion_matrix_normalized.png'),
    ('last.pt — confusion matrix (val)',       'runs/obb/eval_last_val/confusion_matrix_normalized.png'),
    ('last.pt — F1 curve (val)',               'runs/obb/eval_last_val/BoxF1_curve.png'),
]:
    if os.path.exists(path):
        print(f'--- {title} ---')
        display(Image(path))
    else:
        print(f'(нет файла) {path}')

## 9. Своя сводная картинка: train vs val loss + mAP

С вертикальной линией на оптимальной эпохе — готовый слайд для защиты.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('runs/obb/stereo_v1/results.csv')
df.columns = df.columns.str.strip()

train_loss = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
val_loss   = df['val/box_loss']   + df['val/cls_loss']   + df['val/dfl_loss']
best_epoch = int(val_loss.idxmin() + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df['epoch'], train_loss, label='train loss', color='tab:blue')
ax1.plot(df['epoch'], val_loss,   label='val loss',   color='tab:orange')
ax1.axvline(best_epoch, color='green', linestyle='--', label=f'best epoch = {best_epoch}')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (box+cls+dfl)')
ax1.set_title('Train vs Val Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(df['epoch'], df['metrics/mAP50(B)'],    label='mAP50',    color='tab:green')
ax2.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='tab:red')
ax2.axvline(best_epoch, color='green', linestyle='--', alpha=0.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
ax2.set_title('Validation mAP')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('runs/obb/stereo_v1/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Оптимальная эпоха по минимуму val loss: {best_epoch}')

## 10. Скачать всё одним архивом

In [ ]:
!tar -czf /content/stereo_v1_results.tar.gz \
    runs/obb/stereo_v1 \
    runs/obb/eval_best_val \
    runs/obb/eval_best_test \
    runs/obb/eval_last_val \
    runs/obb/eval_last_test
!ls -lh /content/stereo_v1_results.tar.gz

from google.colab import files
files.download('/content/stereo_v1_results.tar.gz')